<a href="https://colab.research.google.com/github/Saurabh312Kumar/Deep_learning/blob/main/ANN_using_hyperparameter_optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from google.colab import drive


In [ ]:
torch.manual_seed(42)

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device : {device}")

using device : cuda


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_df=pd.read_csv("/content/drive/MyDrive/mnist_train.csv")
test_df=pd.read_csv("/content/drive/MyDrive/mnist_test.csv")


In [ ]:
train_df.head()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
X_train = train_df.drop("label", axis=1)
y_train = train_df["label"]

X_test = test_df.drop("label", axis=1)
y_test = test_df["label"]

In [ ]:
print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

(60000, 784)
(60000,)
(10000, 784)
(10000,)


In [ ]:
X_train=X_train/255.0
X_test=X_test/255.0

In [ ]:
# create CustomDataset Class
class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features = torch.tensor(features.values, dtype=torch.float32)
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [ ]:
# create train_dataset object
train_dataset=CustomDataset(X_train,y_train)

In [ ]:
# create train_dataset object
test_dataset=CustomDataset(X_test,y_test)

In [ ]:
class MyNN(nn.Module):

  def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):

    super().__init__()

    layers = []

    for i in range(num_hidden_layers):

      layers.append(nn.Linear(input_dim, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = neurons_per_layer

    layers.append(nn.Linear(neurons_per_layer, output_dim))

    self.model = nn.Sequential(*layers)

  def forward(self, x):

    return self.model(x)

In [ ]:
# objective function
def objective(trial):
  # next hyperparameter values from the search space
  num_hidden_layers=trial.suggest_int("num_hidden_layers",1,5)
  neurons_per_layer=trial.suggest_int("neurons_per_layer",8,128,step=8)
  epochs=trial.suggest_int("epochs",10,100,step=10)
  learning_rate=trial.suggest_float("learning_rate",1e-5, 1e-1, log=True)
  dropout_rate=trial.suggest_float("dropout_rate",0.1, 0.5, step=0.1)
  batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
  optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)


  # create train and test loader
  train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,drop_last=True,pin_memory=True)
  test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

  # model init
  input_dim = 784
  output_dim = 10

  model = MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
  model.to(device)

    # optimizer selection
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.SGD(model.parameters(), lr=0.1, weight_decay=1e-4)

  if optimizer_name == 'Adam':
    optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
    optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
    optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # training loop

  for epoch in range(epochs):

    for batch_features, batch_labels in train_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      # forward pass
      outputs = model(batch_features)

      # calculate loss
      loss = criterion(outputs, batch_labels)

      # back pass
      optimizer.zero_grad()
      loss.backward()

      # update grads
      optimizer.step()


  # evaluation
  model.eval()
  # evaluation on test data
  total = 0
  correct = 0

  with torch.no_grad():

    for batch_features, batch_labels in test_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      outputs = model(batch_features)

      _, predicted = torch.max(outputs, 1)

      total = total + batch_labels.shape[0]

      correct = correct + (predicted == batch_labels).sum().item()

    accuracy = correct/total

  return accuracy

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 26.2 MB/s eta 0:00:00


In [ ]:
import optuna

study = optuna.create_study(direction='maximize')

[I 2026-09-14 15:19:38,972] A new study created in memory with name: no-name-f119769f-29e7-4ff1-bb22-3721e085e268


In [ ]:
study.optimize(objective, n_trials=10)


[I 2026-09-14 15:25:18,039] Trial 0 finished with value: 0.9774 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 48, 'epochs': 50, 'learning_rate': 0.009524012053689688, 'dropout_rate': 0.1, 'batch_size': 64, 'optimizer': 'SGD', 'weight_decay': 0.00012706753272595498}. Best is trial 0 with value: 0.9774.
[I 2026-09-14 15:35:08,528] Trial 1 finished with value: 0.8875 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 8, 'epochs': 70, 'learning_rate': 0.05208487941548883, 'dropout_rate': 0.1, 'batch_size': 16, 'optimizer': 'RMSprop', 'weight_decay': 0.0004631780571618973}. Best is trial 0 with value: 0.9774.
[I 2026-09-14 15:43:05,824] Trial 2 finished with value: 0.9773 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 104, 'epochs': 100, 'learning_rate': 0.00042935868168869927, 'dropout_rate': 0.5, 'batch_size': 64, 'optimizer': 'Adam', 'weight_decay': 0.00018067510497406567}. Best is trial 0 with value: 0.9774.
[I 2026-09-14 15:47:04,121] Trial 3 f

In [ ]:
study.best_value

0.9833

In [ ]:
study.best_params

{'num_hidden_layers': 3,
 'neurons_per_layer': 80,
 'epochs': 80,
 'learning_rate': 2.6383564964371345e-05,
 'dropout_rate': 0.1,
 'batch_size': 128,
 'optimizer': 'Adam',
 'weight_decay': 1.4710663821589047e-05}